# License Plate Recognition (LPR) AI Model

## Project Overview

| Item | Detail |
|------|--------|
| **Goal** | Detect license plates from car images and recognize the text |
| **Framework** | YOLOv8 (Detection) + EasyOCR (Recognition) |
| **Dataset** | Roboflow License Plate Recognition (24,000+ images) |
| **Accuracy** | Detection 90~95% / OCR 85~95% |

---

### Pipeline

```
[Car Image] → [YOLOv8: Find Plate] → [Crop] → [Preprocess] → [EasyOCR: Read Text] → "123가4567"
```

### Setup
1. Runtime → Change runtime type → **T4 GPU** → Save

## Step 1: Install Libraries

| Library | Role |
|---------|------|
| `ultralytics` (YOLOv8) | Detect license plate location in image |
| `easyocr` | Read text from cropped plate image |
| `roboflow` | Download training dataset |
| `cv2` (OpenCV) | Image preprocessing |

In [ ]:
# Install required libraries
!pip install ultralytics -q
!pip install easyocr -q
!pip install roboflow -q

# Import libraries
from ultralytics import YOLO
import easyocr
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files
import os
import torch

# Check GPU
print(f"PyTorch: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU detected. Go to Runtime > Change runtime type > T4 GPU")

## Step 2: Download Dataset (Roboflow)

Using **License Plate Recognition** dataset from Roboflow Universe.

> **YOLO label format:**
> Each image has a `.txt` file with plate location:
> `0 0.45 0.72 0.25 0.08` → (class, center_x, center_y, width, height) all in 0~1 ratio

In [ ]:
from roboflow import Roboflow

# Roboflow API key
rf = Roboflow(api_key="VzTzc1iVXWDK5WSAQJPz")

# Download License Plate Recognition dataset (YOLOv8 format)
project = rf.workspace("roboflow-universe-projects").project("license-plate-recognition-rxg4e")
version = project.version(4)
dataset = version.download("yolov8")

print(f"\nDataset downloaded to: {dataset.location}")
print(f"Train images: {dataset.location}/train/images/")
print(f"Valid images: {dataset.location}/valid/images/")

## Step 3: Explore Dataset

Check what the training images and labels look like.

In [ ]:
# Display sample training images
train_img_dir = f"{dataset.location}/train/images/"
train_images = os.listdir(train_img_dir)
print(f"Total training images: {len(train_images)}")

# Show 8 sample images
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Training Data Samples', fontsize=16)

for i, ax in enumerate(axes.flat):
    if i < len(train_images):
        img_path = os.path.join(train_img_dir, train_images[i])
        img = cv2.imread(img_path)
        if img is not None:
            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
            ax.set_title(train_images[i][:20], fontsize=8)
    ax.axis('off')

plt.tight_layout()
plt.show()

# Check label format
train_label_dir = f"{dataset.location}/train/labels/"
label_files = os.listdir(train_label_dir)
print(f"\nTotal label files: {len(label_files)}")
if label_files:
    sample_label = os.path.join(train_label_dir, label_files[0])
    with open(sample_label, 'r') as f:
        print(f"Sample label ({label_files[0]}):")
        print(f"  {f.read().strip()}")

## Step 4: Train YOLOv8 Plate Detection Model

**Fine-tuning:** Take a pre-trained YOLOv8 model (trained on millions of images)
and train it specifically on license plate images.

| Model | Parameters | Speed | Accuracy |
|-------|-----------|-------|----------|
| YOLOv8n (nano) | 3.2M | Very fast | Good |
| YOLOv8s (small) | 11.2M | Fast | Better |
| YOLOv8m (medium) | 25.9M | Medium | Best |

> Training takes about **30~60 minutes** with T4 GPU (50 epochs).

In [ ]:
# Load pre-trained YOLOv8 nano model
model = YOLO('yolov8n.pt')

# Train on license plate dataset
print("Training started...\n")
results = model.train(
    data=f"{dataset.location}/data.yaml",  # Dataset config
    epochs=50,          # 50 training cycles
    imgsz=640,          # Input image size
    batch=16,           # Process 16 images at a time
    name='plate_detect' # Output folder name
)

print("\nTraining complete!")

## Step 5: Evaluate Training Results

Check training metrics:
- **mAP (mean Average Precision):** Higher is better (target: 90%+)
- **Loss:** Lower is better

In [ ]:
# Display training results
from IPython.display import Image as IPImage, display

# Training curves
results_img = 'runs/detect/plate_detect/results.png'
if os.path.exists(results_img):
    display(IPImage(filename=results_img, width=800))
    print("Training curves: Loss should decrease, mAP should increase")

# Confusion matrix
cm_img = 'runs/detect/plate_detect/confusion_matrix.png'
if os.path.exists(cm_img):
    display(IPImage(filename=cm_img, width=500))

# Validation predictions
val_img = 'runs/detect/plate_detect/val_batch0_pred.png'
if os.path.exists(val_img):
    display(IPImage(filename=val_img, width=800))
    print("Validation predictions with bounding boxes")

## Step 6: Test Detection on Sample Images

Load the best trained model and test it on validation images.

In [ ]:
# Load the best trained model
best_model = YOLO('runs/detect/plate_detect/weights/best.pt')

# Test on validation images
val_img_dir = f"{dataset.location}/valid/images/"
val_images = os.listdir(val_img_dir)

# Show detection results for 6 images
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('License Plate Detection Results', fontsize=16)

for i, ax in enumerate(axes.flat):
    if i < len(val_images):
        img_path = os.path.join(val_img_dir, val_images[i])
        results = best_model(img_path, verbose=False)
        result_img = results[0].plot()
        ax.imshow(cv2.cvtColor(result_img, cv2.COLOR_BGR2RGB))

        # Show confidence
        confs = [f"{c:.0%}" for c in results[0].boxes.conf.cpu().numpy()]
        ax.set_title(f"Conf: {', '.join(confs)}" if confs else "No detection", fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.show()

## Step 7: Crop & Preprocess Detected Plates

After detecting the plate location, we:
1. **Crop** the plate region from the image
2. **Grayscale** conversion
3. **CLAHE** contrast enhancement
4. **Denoise** to remove noise

This makes OCR much more accurate.

In [ ]:
def crop_and_preprocess(image_path, model, confidence_threshold=0.5):
    """
    Detect license plate, crop it, and preprocess for OCR.

    Returns:
        list of dicts with 'image', 'preprocessed', 'confidence', 'bbox'
    """
    img = cv2.imread(image_path)
    results = model(image_path, verbose=False)
    plates = []

    for box in results[0].boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0].cpu().numpy())
        confidence = float(box.conf[0].cpu().numpy())

        if confidence < confidence_threshold:
            continue

        # Crop plate region
        plate_img = img[y1:y2, x1:x2]
        if plate_img.size == 0:
            continue

        # Preprocess
        gray = cv2.cvtColor(plate_img, cv2.COLOR_BGR2GRAY)

        # Resize (width = 300px)
        h, w = gray.shape
        new_w = 300
        new_h = int(h * (new_w / w))
        resized = cv2.resize(gray, (new_w, new_h))

        # CLAHE contrast enhancement
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        enhanced = clahe.apply(resized)

        # Denoise
        denoised = cv2.fastNlMeansDenoising(enhanced, h=10)

        plates.append({
            'image': plate_img,
            'preprocessed': denoised,
            'confidence': confidence,
            'bbox': (x1, y1, x2, y2)
        })

    return plates


# Test: crop and preprocess
test_img = os.path.join(val_img_dir, val_images[0])
plates = crop_and_preprocess(test_img, best_model)

if plates:
    fig, axes = plt.subplots(len(plates), 2, figsize=(12, 4 * len(plates)))
    if len(plates) == 1:
        axes = [axes]
    for i, plate in enumerate(plates):
        axes[i][0].imshow(cv2.cvtColor(plate['image'], cv2.COLOR_BGR2RGB))
        axes[i][0].set_title(f"Cropped (conf: {plate['confidence']:.1%})")
        axes[i][0].axis('off')
        axes[i][1].imshow(plate['preprocessed'], cmap='gray')
        axes[i][1].set_title('Preprocessed (grayscale + CLAHE + denoise)')
        axes[i][1].axis('off')
    plt.tight_layout()
    plt.show()
else:
    print("No plates detected in this image.")

## Step 8: OCR - Read Plate Text (EasyOCR)

EasyOCR uses deep learning to recognize text in images.
We use Korean (`ko`) + English (`en`) language support.

In [ ]:
# Initialize EasyOCR reader (Korean + English)
print("Loading EasyOCR model (first time takes ~1 min)...")
reader = easyocr.Reader(['ko', 'en'], gpu=True)
print("EasyOCR ready!")


def recognize_plate_text(plate_image, reader):
    """
    Read text from preprocessed plate image using EasyOCR.

    Returns:
        full_text: combined plate text
        details: list of (text, confidence) tuples
    """
    results = reader.readtext(plate_image)

    details = []
    for (bbox, text, confidence) in results:
        details.append({'text': text, 'confidence': confidence})

    full_text = ' '.join([d['text'] for d in details])
    return full_text, details


# Test OCR on detected plates
if plates:
    for i, plate in enumerate(plates):
        text, details = recognize_plate_text(plate['preprocessed'], reader)
        print(f"Plate {i+1}: '{text}'")
        for d in details:
            print(f"  - '{d['text']}' (confidence: {d['confidence']:.1%})")

## Step 9: Complete Pipeline

Combine all steps into one function:

```
Image → YOLOv8 Detection → Crop → Preprocess → EasyOCR → Plate Text
```

In [ ]:
def recognize_license_plate(image_path, yolo_model, ocr_reader):
    """
    Full pipeline: detect plate -> crop -> preprocess -> OCR

    Args:
        image_path: path to car image
        yolo_model: trained YOLOv8 model
        ocr_reader: EasyOCR reader

    Returns:
        list of recognized plates with text and confidence
    """
    img = cv2.imread(image_path)
    if img is None:
        print(f"Cannot read image: {image_path}")
        return []

    # Step 1: Detect plates
    detections = yolo_model(image_path, verbose=False)
    results = []

    for box in detections[0].boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0].cpu().numpy())
        det_conf = float(box.conf[0].cpu().numpy())

        if det_conf < 0.5:
            continue

        # Step 2: Crop
        plate_img = img[y1:y2, x1:x2]
        if plate_img.size == 0:
            continue

        # Step 3: Preprocess
        gray = cv2.cvtColor(plate_img, cv2.COLOR_BGR2GRAY)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        enhanced = clahe.apply(gray)
        denoised = cv2.fastNlMeansDenoising(enhanced, h=10)

        # Step 4: OCR
        ocr_results = ocr_reader.readtext(denoised)
        plate_text = ' '.join([text for _, text, _ in ocr_results])
        ocr_conf = np.mean([conf for _, _, conf in ocr_results]) if ocr_results else 0

        results.append({
            'text': plate_text,
            'detection_confidence': det_conf,
            'ocr_confidence': ocr_conf,
            'bbox': (x1, y1, x2, y2),
            'plate_image': plate_img,
            'preprocessed': denoised
        })

    return results


# Test full pipeline on multiple validation images
print("=" * 50)
print("  Full Pipeline Test")
print("=" * 50)

test_count = min(6, len(val_images))
fig, axes = plt.subplots(test_count, 3, figsize=(18, 5 * test_count))
if test_count == 1:
    axes = [axes]

for i in range(test_count):
    img_path = os.path.join(val_img_dir, val_images[i])
    results = recognize_license_plate(img_path, best_model, reader)

    # Original image with bounding box
    img = cv2.imread(img_path)
    for r in results:
        x1, y1, x2, y2 = r['bbox']
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 3)
    axes[i][0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    axes[i][0].set_title('Detection', fontsize=10)
    axes[i][0].axis('off')

    if results:
        # Cropped plate
        axes[i][1].imshow(cv2.cvtColor(results[0]['plate_image'], cv2.COLOR_BGR2RGB))
        axes[i][1].set_title(f"Cropped (det: {results[0]['detection_confidence']:.0%})", fontsize=10)
        axes[i][1].axis('off')

        # Preprocessed + OCR result
        axes[i][2].imshow(results[0]['preprocessed'], cmap='gray')
        axes[i][2].set_title(f"OCR: {results[0]['text']} ({results[0]['ocr_confidence']:.0%})", fontsize=10)
        axes[i][2].axis('off')

        print(f"Image {i+1}: '{results[0]['text']}' (det: {results[0]['detection_confidence']:.0%}, ocr: {results[0]['ocr_confidence']:.0%})")
    else:
        axes[i][1].text(0.5, 0.5, 'No plate detected', ha='center', va='center')
        axes[i][1].axis('off')
        axes[i][2].axis('off')
        print(f"Image {i+1}: No plate detected")

plt.tight_layout()
plt.show()

## Step 10: Upload Your Own Photo

Test with your own car photos!

**Tips for best results:**
- Plate should be clearly visible
- Front or rear view of the car
- Not too far away (plate should be readable by human eye)

In [ ]:
# Upload your own car photos
print("Upload car images (PNG/JPG):")
print("Tip: plate should be clearly visible.\n")
uploaded = files.upload()

for filename in uploaded.keys():
    filepath = f'/content/{filename}'
    with open(filepath, 'wb') as f:
        f.write(uploaded[filename])

    print(f"\n{'='*50}")
    print(f"  {filename}")
    print(f"{'='*50}")

    # Run full pipeline
    results = recognize_license_plate(filepath, best_model, reader)

    if results:
        # Visualize
        img = cv2.imread(filepath)
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))

        # Original + bbox
        for r in results:
            x1, y1, x2, y2 = r['bbox']
            cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 3)
        axes[0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        axes[0].set_title('Detection', fontsize=12)
        axes[0].axis('off')

        # Cropped plate
        axes[1].imshow(cv2.cvtColor(results[0]['plate_image'], cv2.COLOR_BGR2RGB))
        axes[1].set_title(f"Plate (conf: {results[0]['detection_confidence']:.0%})", fontsize=12)
        axes[1].axis('off')

        # Preprocessed
        axes[2].imshow(results[0]['preprocessed'], cmap='gray')
        axes[2].set_title('Preprocessed', fontsize=12)
        axes[2].axis('off')

        plt.tight_layout()
        plt.show()

        # Print results
        for i, r in enumerate(results):
            print(f"\n  Plate {i+1}: {r['text']}")
            print(f"    Detection confidence: {r['detection_confidence']:.1%}")
            print(f"    OCR confidence: {r['ocr_confidence']:.1%}")
    else:
        # Show original image
        img = cv2.imread(filepath)
        plt.figure(figsize=(10, 6))
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        plt.title('No plate detected')
        plt.axis('off')
        plt.show()
        print("  No license plate detected.")

## Step 11: Save Model to Google Drive

Save the trained model permanently to Google Drive.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import shutil

# Save trained YOLOv8 model
save_dir = '/content/drive/MyDrive/license_plate_model/'
os.makedirs(save_dir, exist_ok=True)

model_src = 'runs/detect/plate_detect/weights/best.pt'
model_dst = f'{save_dir}/yolov8_plate_detector.pt'

shutil.copy(model_src, model_dst)
print(f"Model saved: {model_dst}")
print(f"\nTo load later:")
print(f"  model = YOLO('{model_dst}')")

---

## Glossary

| Term | Description |
|------|-------------|
| Object Detection | Finding location and type of objects in an image |
| YOLO | You Only Look Once - real-time object detection model |
| OCR | Optical Character Recognition - reading text from images |
| Bounding Box | Rectangle around a detected object |
| Fine-tuning | Additional training of a pre-trained model on specific data |
| mAP | mean Average Precision - detection accuracy metric |
| CLAHE | Contrast Limited Adaptive Histogram Equalization |
| Crop | Cutting out a specific region from an image |
| Pipeline | Multiple processing steps connected in sequence |

---

## Next Steps

| Level | Task | Difficulty |
|-------|------|------------|
| 1 | Night/rainy plate recognition (Data Augmentation) | ★★★☆ |
| 2 | Real-time video plate recognition (webcam) | ★★★☆ |
| 3 | Plate angle correction (Perspective Transform) | ★★★★ |
| 4 | Parking lot entry/exit auto-logging system | ★★★★ |
| 5 | Edge AI deployment (Raspberry Pi + Camera) | ★★★★★ |